# TEST

In [2]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import re

# ✅ run 폴더
run_dir = Path("../runs/exp1_qwen2p5_baseline_20260226_101302")

# ✅ harness report 파일명
harness_report = Path("../exp1-step2-4-qwen-200_1024.json")

print("run_dir exists:", run_dir.exists(), run_dir)
print("results.csv exists:", (run_dir/"results.csv").exists())
print("traces dir exists:", (run_dir/"traces").exists())
print("harness report exists:", harness_report.exists(), harness_report)

run_dir exists: True ../runs/exp1_qwen2p5_baseline_20260226_101302
results.csv exists: True
traces dir exists: True
harness report exists: True ../exp1-step2-4-qwen-200_1024.json


In [3]:
df = pd.read_csv(run_dir/"results.csv")
print("Total rows:", len(df))
display(df.head(3))

# 타입 정리(가끔 trial_id가 object로 들어옴)
df["trial_id"] = pd.to_numeric(df["trial_id"], errors="coerce").fillna(0).astype(int)
df["files_changed"] = pd.to_numeric(df.get("files_changed"), errors="coerce")
df["patch_lines_added"] = pd.to_numeric(df.get("patch_lines_added"), errors="coerce")
df["patch_lines_removed"] = pd.to_numeric(df.get("patch_lines_removed"), errors="coerce")

# diff "크기" 지표
df["diff_total_lines"] = (df["patch_lines_added"].fillna(0) + df["patch_lines_removed"].fillna(0)).astype(int)

Total rows: 200


,task_id,trial_id,model,prompt_hash,taxonomy_version,success,stage,error_type,signature,returncode,...,format_used,format_ok,format_reason,apply_check_ok,apply_check_reason,patch_lines_added,patch_lines_removed,files_changed,timestamp,seed
0,astropy__astropy-12907,0,Qwen/Qwen2.5-Coder-7B-Instruct,1d006f36813d2fc095bc8da160e7e87abfd8d05ee8957d...,step2-4,False,EXEC,EXEC_FAIL,docker_image_not_found,125.0,...,NaN,NaN,NaN,NaN,NaN,4,1,1,2026-02-26T10:13:02.250924,42
1,astropy__astropy-14182,0,Qwen/Qwen2.5-Coder-7B-Instruct,323a3b727d26875c53e665ce8dcd10f792793ee5799a7a...,step2-4,False,EDIT_PARSE,GEN_FAIL,invalid_edit_script,-1.0,...,NaN,NaN,NaN,NaN,NaN,0,0,0,2026-02-26T10:13:07.087934,42
2,astropy__astropy-14365,0,Qwen/Qwen2.5-Coder-7B-Instruct,232a841813d672f784b09b1e96ec3c80e68dec2004a896...,step2-4,False,EXEC,EXEC_FAIL,docker_image_not_found,125.0,...,NaN,NaN,NaN,NaN,NaN,7,1,1,2026-02-26T10:13:17.954124,42


In [4]:
traces_dir = run_dir/"traces"

def safe_read(p: Path):
    try:
        return p.read_text(encoding="utf-8", errors="ignore")
    except Exception:
        return None

# trace_path 만들기: {task_id}_trial{trial}.json
def trace_json_path(task_id, trial_id):
    return traces_dir / f"{task_id}_trial{trial_id}.json"

def edit_json_path(task_id, trial_id):
    return traces_dir / f"{task_id}_trial{trial_id}.edit.json"

def diff_path(task_id, trial_id):
    return traces_dir / f"{task_id}_trial{trial_id}.patch.diff"

# traces에서 필요한 필드만 추출
rows = []
for _, r in df.iterrows():
    tid = r["task_id"]
    tr = r["trial_id"]
    tj = trace_json_path(tid, tr)
    ej = edit_json_path(tid, tr)
    dp = diff_path(tid, tr)

    trace_txt = safe_read(tj)
    edit_txt = safe_read(ej)
    diff_txt = safe_read(dp)

    stderr_trace = None
    stdout_trace = None
    issue_text = None
    test_command = None

    if trace_txt:
        try:
            obj = json.loads(trace_txt)
            stderr_trace = obj.get("stderr")
            stdout_trace = obj.get("stdout")
            issue_text = obj.get("issue_text")
            test_command = obj.get("test_command")
        except Exception:
            pass

    rows.append({
        "task_id": tid,
        "trial_id": tr,
        "trace_path": str(tj) if tj.exists() else None,
        "stderr_trace": stderr_trace,
        "stdout_trace": stdout_trace,
        "issue_text": issue_text,
        "test_command_trace": test_command,
        "edit_script_trace": edit_txt,
        "diff_trace": diff_txt,
    })

df_tr = pd.DataFrame(rows)
df_join = df.merge(df_tr, on=["task_id","trial_id"], how="left")

print("Joined rows:", len(df_join))
display(df_join.head(3))

Joined rows: 200


,task_id,trial_id,model,prompt_hash,taxonomy_version,success,stage,error_type,signature,returncode,...,timestamp,seed,diff_total_lines,trace_path,stderr_trace,stdout_trace,issue_text,test_command_trace,edit_script_trace,diff_trace
0,astropy__astropy-12907,0,Qwen/Qwen2.5-Coder-7B-Instruct,1d006f36813d2fc095bc8da160e7e87abfd8d05ee8957d...,step2-4,False,EXEC,EXEC_FAIL,docker_image_not_found,125.0,...,2026-02-26T10:13:02.250924,42,5,../runs/exp1_qwen2p5_baseline_20260226_101302/...,Unable to find image 'swebench/sweb.eval.x86_6...,,None,echo 'No test command',"{\n ""edits"": [\n {\n ""op""...",diff --git a/astropy/modeling/core.py b/astrop...
1,astropy__astropy-14182,0,Qwen/Qwen2.5-Coder-7B-Instruct,323a3b727d26875c53e665ce8dcd10f792793ee5799a7a...,step2-4,False,EDIT_PARSE,GEN_FAIL,invalid_edit_script,-1.0,...,2026-02-26T10:13:07.087934,42,0,../runs/exp1_qwen2p5_baseline_20260226_101302/...,"Expecting ',' delimiter: line 8 column 6 (char...",,None,,"{\n ""edits"": [\n {\n ""op"": ""insert_af...",None
2,astropy__astropy-14365,0,Qwen/Qwen2.5-Coder-7B-Instruct,232a841813d672f784b09b1e96ec3c80e68dec2004a896...,step2-4,False,EXEC,EXEC_FAIL,docker_image_not_found,125.0,...,2026-02-26T10:13:17.954124,42,8,../runs/exp1_qwen2p5_baseline_20260226_101302/...,Unable to find image 'swebench/sweb.eval.x86_6...,,None,echo 'No test command',"{\n ""edits"": [\n {\n ""op"": ""replace_r...",diff --git a/astropy/io/ascii/qdp.py b/astropy...


In [5]:
with open(harness_report, "r", encoding="utf-8") as f:
    rep = json.load(f)

# harness report는 schema_version 2에서 resolved_ids / unresolved_ids / error_ids 같은 리스트가 있음
resolved = set(rep.get("resolved_ids", []) or [])
unresolved = set(rep.get("unresolved_ids", []) or [])
error_ids = set(rep.get("error_ids", []) or [])
submitted = set(rep.get("submitted_ids", []) or [])

# not_submitted = dataset 전체 - submitted
# dataset 전체 id 목록은 report에 없을 수 있으니, 여기서는 df_join 기준으로만 표시
def harness_status(iid: str):
    if iid in resolved: return "resolved"
    if iid in error_ids: return "error"
    if iid in unresolved: return "unresolved"
    if iid in submitted: return "submitted_unknown"
    return "not_submitted"

df_join["harness_status"] = df_join["task_id"].map(harness_status)

display(df_join["harness_status"].value_counts().to_frame("count"))

,count
harness_status,
unresolved,153
not_submitted,46
resolved,1


### edit op 유형 파싱( insert_after / replace_range )

In [6]:
def parse_ops(edit_script_text: str):
    """
    returns (ok, ops_list)
    """
    if not isinstance(edit_script_text, str) or not edit_script_text.strip():
        return False, []
    try:
        obj = json.loads(edit_script_text)
        edits = obj.get("edits", [])
        if not isinstance(edits, list):
            return False, []
        ops = []
        for e in edits:
            if isinstance(e, dict) and "op" in e:
                ops.append(str(e["op"]))
        return True, ops
    except Exception:
        return False, []

ops_ok = []
ops_list = []
n_edits = []
for s in df_join["edit_script_trace"].tolist():
    ok, ops = parse_ops(s)
    ops_ok.append(ok)
    ops_list.append(ops)
    n_edits.append(len(ops))

df_join["ops_parse_ok"] = ops_ok
df_join["ops_list"] = ops_list
df_join["n_edits"] = n_edits

# op one-hot(빈도 분석용)
all_ops = sorted({op for ops in ops_list for op in ops})
for op in all_ops:
    df_join[f"op__{op}"] = df_join["ops_list"].apply(lambda xs: int(op in xs))

print("Detected ops:", all_ops)
display(df_join[["task_id","harness_status","n_edits","ops_list"]].head(5))

Detected ops: ['insert_after', 'replace_range']


,task_id,harness_status,n_edits,ops_list
0,astropy__astropy-12907,unresolved,1,[insert_after]
1,astropy__astropy-14182,not_submitted,0,[]
2,astropy__astropy-14365,unresolved,2,"[replace_range, insert_after]"
3,astropy__astropy-14995,unresolved,1,[replace_range]
4,astropy__astropy-6938,not_submitted,1,[replace_range]


In [7]:
target = df_join[df_join["harness_status"].isin(["resolved","unresolved"])].copy()
print("resolved:", (target["harness_status"]=="resolved").sum(),
      "unresolved:", (target["harness_status"]=="unresolved").sum())

num_cols = [
    "diff_total_lines",
    "patch_lines_added","patch_lines_removed","files_changed",
    "context_num_files",
    "n_edits",
]

# 없는 컬럼 대비
num_cols = [c for c in num_cols if c in target.columns]

summary = target.groupby("harness_status")[num_cols].agg(["count","mean","median","std","min","max"])
display(summary)

resolved: 1 unresolved: 153


diff_total_lines                                     \
                          count      mean median       std min max   
harness_status                                                       
resolved                      1  6.000000    6.0       NaN   6   6   
unresolved                  153  6.797386    5.0  5.166134   1  32   

               patch_lines_added                             ...  \
                           count      mean median       std  ...   
harness_status                                               ...   
resolved                       1  2.000000    2.0       NaN  ...   
unresolved                   153  4.836601    3.0  5.185325  ...   

               context_num_files                   n_edits                   \
                          median       std min max   count      mean median   
harness_status                                                                
resolved                    80.0       NaN  80  80       1  1.000000    1.0   
unresolved                  80.0  0.515998  75  80     153  1.509804    1.0   

                                  
                     std min max  
harness_status                    
resolved             NaN   1   1  
unresolved      0.795693   1   8  

[2 rows x 36 columns]

In [8]:
def describe_diff(col):
    a = target[target["harness_status"]=="resolved"][col].dropna()
    b = target[target["harness_status"]=="unresolved"][col].dropna()
    if len(a)==0 or len(b)==0:
        return None
    return {
        "col": col,
        "resolved_value(s)": a.tolist(),
        "unresolved_mean": float(b.mean()),
        "unresolved_median": float(b.median()),
        "unresolved_std": float(b.std()) if len(b)>1 else 0.0
    }

rows = []
for c in num_cols:
    d = describe_diff(c)
    if d: rows.append(d)

display(pd.DataFrame(rows))

,col,resolved_value(s),unresolved_mean,unresolved_median,unresolved_std
0,diff_total_lines,[6],6.797386,5.0,5.166134
1,patch_lines_added,[2],4.836601,3.0,5.185325
2,patch_lines_removed,[4],1.960784,2.0,1.394927
3,files_changed,[1],1.013072,1.0,0.113956
4,context_num_files,[80],79.941176,80.0,0.515998
5,n_edits,[1],1.509804,1.0,0.795693


## Resolved 살펴보기

In [14]:
sol = df_join[df_join["harness_status"]=="resolved"].copy()
print("resolved rows:", len(sol))
if len(sol):
    r = sol.iloc[0]
    print("TASK:", r["task_id"], "trial:", r["trial_id"])
    print("\n== ops_list ==")
    print(r.get("ops_list"))
    print("\n== n_edits ==")
    print(r.get("n_edits"))
    print("\n== diff stats ==")
    print("added:", r.get("patch_lines_added"), "removed:", r.get("patch_lines_removed"), "files_changed:", r.get("files_changed"))
    print("\n== diff preview ==")
    print((r.get("diff_trace") or "")[:2000])
    print("\n== stderr preview ==")
    print((r.get("stderr_trace") or "")[:1000])

resolved rows: 1
TASK: psf__requests-863 trial: 0

== ops_list ==
['replace_range']

== n_edits ==
1

== diff stats ==
added: 2 removed: 4 files_changed: 1

== diff preview ==
diff --git a/requests/hooks.py b/requests/hooks.py
index 9e0ce346..0ad82b92 100644
--- a/requests/hooks.py
+++ b/requests/hooks.py
@@ -17,10 +17,8 @@ Available hooks:
 ``pre_send``:
     The Request object, directly before being sent.
 
-``post_request``:
-    The Request object, directly after being sent.
-
-``response``:
+        for name, hooks in hooks.items():
+            self._hooks[name] = [hook for hook in hooks if callable(hook)]``response``:
     The response generated from a Request.
 
 """


== stderr preview ==
Unable to find image 'swebench/sweb.eval.x86_64:latest' locally
docker: Error response from daemon: pull access denied for swebench/sweb.eval.x86_64, repository does not exist or may require 'docker login'

Run 'docker run --help' for more information



### Unresolved 살펴보기

In [15]:
uns = df_join[df_join["harness_status"]=="unresolved"].copy()
if len(uns):
    display(uns.sort_values("diff_total_lines", ascending=False)[["task_id","diff_total_lines","files_changed","n_edits","ops_list","signature","stage"]].head(5))
    display(uns.sort_values("diff_total_lines", ascending=True)[["task_id","diff_total_lines","files_changed","n_edits","ops_list","signature","stage"]].head(5))

,task_id,diff_total_lines,files_changed,n_edits,ops_list,signature,stage
46,django__django-13028,32,1,1,[replace_range],docker_image_not_found,EXEC
107,django__django-16229,31,1,1,[insert_after],docker_image_not_found,EXEC
69,django__django-14155,21,1,3,"[replace_range, insert_after, replace_range]",docker_image_not_found,EXEC
48,django__django-13158,21,1,2,"[replace_range, insert_after]",docker_image_not_found,EXEC
66,django__django-13964,21,2,2,"[insert_after, insert_after]",docker_image_not_found,EXEC


,task_id,diff_total_lines,files_changed,n_edits,ops_list,signature,stage
74,django__django-14580,1,1,1,[insert_after],docker_image_not_found,EXEC
153,psf__requests-2674,1,1,1,[insert_after],docker_image_not_found,EXEC
89,django__django-15320,1,1,1,[insert_after],docker_image_not_found,EXEC
85,django__django-15061,1,1,1,[replace_range],docker_image_not_found,EXEC
128,matplotlib__matplotlib-23913,1,1,1,[insert_after],docker_image_not_found,EXEC
